# 8.6 Feature Selection vs Feature Extraction

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 8: Dimensionality Reduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-08-dimensionality-reduction/08.06_feature-selection-vs-feature-extraction.ipynb)

Companion code for Section 8.6 of the book. Run the cells in order from top to bottom.

**Example 8.6.1**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression, LassoCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, f_classif, mutual_info_classif,
    RFE, RFECV, SelectFromModel, chi2
)
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
import warnings; warnings.filterwarnings('ignore')

data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
X_sc = StandardScaler().fit_transform(X)

print('=== Feature Selection vs Feature Extraction Comparison ===')
print(f'Original: {X.shape[1]} features')
print(f'{"Method":<35} {"Features":>10} {"CV Accuracy":>14} {"Std":>8}')
print('='*72)

# ── Baseline: all features ────────────────────────────────────────
lr_base = LogisticRegression(max_iter=1000, random_state=42)
sc_base = cross_val_score(lr_base, X_sc, y, cv=5, scoring='accuracy')
print(f'{"All features (baseline)":<35} {X.shape[1]:>10} {sc_base.mean():>14.4f} {sc_base.std():>8.4f}')

# ── Feature Selection Methods ─────────────────────────────────────

# 1. Variance Threshold
var_sel = VarianceThreshold(threshold=0.1)
X_var = var_sel.fit_transform(X_sc)
sc = cross_val_score(Pipeline([('lr',lr_base)]), X_var, y, cv=5, scoring='accuracy')
print(f'{"Variance Threshold (>0.1)":<35} {X_var.shape[1]:>10} {sc.mean():>14.4f} {sc.std():>8.4f}')

# 2. SelectKBest with ANOVA F-test
for k in [5, 10, 15]:
    pipe = Pipeline([('sel', SelectKBest(f_classif, k=k)), ('lr', lr_base)])
    sc = cross_val_score(pipe, X_sc, y, cv=5, scoring='accuracy')
    print(f'{f"SelectKBest ANOVA (k={k})":<35} {k:>10} {sc.mean():>14.4f} {sc.std():>8.4f}')

# 3. Mutual Information
pipe_mi = Pipeline([('sel', SelectKBest(mutual_info_classif, k=10)), ('lr', lr_base)])
sc_mi = cross_val_score(pipe_mi, X_sc, y, cv=5, scoring='accuracy')
print(f'{"Mutual Info (k=10)":<35} {10:>10} {sc_mi.mean():>14.4f} {sc_mi.std():>8.4f}')

# 4. RFE with Logistic Regression
rfe = RFE(estimator=LogisticRegression(max_iter=1000, random_state=42), n_features_to_select=10)
pipe_rfe = Pipeline([('rfe', rfe), ('lr', lr_base)])
sc_rfe = cross_val_score(pipe_rfe, X_sc, y, cv=5, scoring='accuracy')
print(f'{"RFE (n=10 features)":<35} {10:>10} {sc_rfe.mean():>14.4f} {sc_rfe.std():>8.4f}')

# 5. Lasso-based (embedded)
lasso = LassoCV(cv=5, random_state=42).fit(X_sc, y)
sfm_lasso = SelectFromModel(lasso, prefit=True)
X_lasso = sfm_lasso.transform(X_sc)
sc_las = cross_val_score(lr_base, X_lasso, y, cv=5, scoring='accuracy')
print(f'{"Lasso feature selection":<35} {X_lasso.shape[1]:>10} {sc_las.mean():>14.4f} {sc_las.std():>8.4f}')

# 6. RandomForest importance
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_sc, y)
sfm_rf = SelectFromModel(rf, prefit=True, threshold='median')
X_rf = sfm_rf.transform(X_sc)
sc_rf_sel = cross_val_score(lr_base, X_rf, y, cv=5, scoring='accuracy')
print(f'{"RF importance (>median)":<35} {X_rf.shape[1]:>10} {sc_rf_sel.mean():>14.4f} {sc_rf_sel.std():>8.4f}')

print()

# ── Feature Extraction Methods ────────────────────────────────────
for k in [2, 5, 10, 15, 20]:
    pipe = Pipeline([('pca', PCA(n_components=k)), ('lr', lr_base)])
    sc = cross_val_score(pipe, X_sc, y, cv=5, scoring='accuracy')
    print(f'{f"PCA (k={k} components)":<35} {k:>10} {sc.mean():>14.4f} {sc.std():>8.4f}')

# ── Which features are most selected? ─────────────────────────────
print('\n=== Feature Importance Ranking ===')
# Collect rankings from multiple methods
rankings = {}

# ANOVA F-scores
f_scores, _ = f_classif(X_sc, y)
rankings['ANOVA F-score'] = f_scores / f_scores.max()

# Mutual information
mi_scores = mutual_info_classif(X_sc, y, random_state=42)
rankings['Mutual Info'] = mi_scores / mi_scores.max()

# RF importance
rankings['RF Importance'] = rf.feature_importances_ / rf.feature_importances_.max()

# RFE ranking
rfe_fitted = RFE(LogisticRegression(max_iter=1000,random_state=42), n_features_to_select=1)
rfe_fitted.fit(X_sc, y)
rankings['RFE Ranking'] = 1 - (rfe_fitted.ranking_ - 1) / rfe_fitted.ranking_.max()

rank_df = pd.DataFrame(rankings, index=feature_names)
rank_df['Mean Rank'] = rank_df.mean(axis=1)
rank_df = rank_df.sort_values('Mean Rank', ascending=False)

print('Top 10 features (consensus ranking across 4 methods):')
print(rank_df.head(10).round(4).to_string())

# Visualise
fig, ax = plt.subplots(figsize=(12, 7))
rank_df.head(10)[['ANOVA F-score','Mutual Info','RF Importance','RFE Ranking']].plot(
    kind='bar', ax=ax, colormap='Set2', edgecolor='white')
ax.set_title('Feature Importance - 4 Methods Compared (Top 10 Features)',
            fontsize=13, fontweight='bold')
ax.set_xlabel('Feature'); ax.set_ylabel('Normalised Score')
ax.legend(fontsize=10); ax.grid(alpha=0.3, axis='y')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig('feature_selection_comparison.png', dpi=150, bbox_inches='tight')
plt.show()